# Oral exam demo — Exploring the Silver tables

This notebook opens the **existing Parquet outputs** without rerunning the pipeline or modifying the data.

In VS Code, select the VM's Python kernel, ideally `/home/ubuntu/spark-env/bin/python`, then choose **Run All** or use **Shift + Enter** to run one cell at a time.

We use the **latest validated Silver run** in the `dev` environment. Small previews are displayed with `.show()` to avoid loading entire tables into Python memory.

## 1. Find the latest validated run

`latest_successful.json` points to the run directory. The report must have status `passed` and the same run ID as the pointer.

In [1]:
import json
import os
import sys
from pathlib import Path

# Works from the project root or from notebooks/.
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "config/dev.json").is_file()
)
environment = "dev"
settings = json.loads((root / "config" / f"{environment}.json").read_text())
silver_root = root / settings["silver"]

pointer = json.loads((silver_root / "latest_successful.json").read_text())
run_path = Path(pointer["path"])
report = json.loads((run_path / "quality_report.json").read_text())
assert report["status"] == "passed", "This Silver run has not been validated."
assert report["run_id"] == pointer["run_id"], "The pointer and report do not match."

print("Environment:", environment)
print("Silver run:", report["run_id"])
print("Directory:", run_path)
print("Status:", report["status"])

Environment: dev
Silver run: 20261115T161713385306Z
Directory: /home/ubuntu/group16/project/data/silver/dev/runs/20261115T161713385306Z
Status: passed


## 2. Start Spark and load the four tables

A **DataFrame** represents an entire table. Spark reads each table's directory and combines its `part-….parquet` files.

We reuse the VM's Spark installation without installing any additional packages.

In [2]:
os.environ.setdefault("SPARK_HOME", "/opt/spark")
os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-8-openjdk-amd64")
os.environ["PYSPARK_PYTHON"] = sys.executable

# Spark Python libraries may be bundled with /opt/spark rather than installed via pip.
spark_python = Path(os.environ["SPARK_HOME"]) / "python"
if spark_python.is_dir():
    sys.path.insert(0, str(spark_python))
    for archive in sorted((spark_python / "lib").glob("*.zip")):
        sys.path.insert(0, str(archive))

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Silver oral exam demo")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.session.timeZone", "Europe/London")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

collisions = spark.read.parquet(str(run_path / "collisions"))
vehicles = spark.read.parquet(str(run_path / "vehicles"))
casualties = spark.read.parquet(str(run_path / "casualties"))
enriched = spark.read.parquet(str(run_path / "collisions_enriched"))

tables = {
    "collisions": collisions,
    "vehicles": vehicles,
    "casualties": casualties,
    "collisions_enriched": enriched,
}
print("Spark:", spark.version)
print("Tables loaded:", ", ".join(tables))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 19:22:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



[Stage 0:>                                                          (0 + 1) / 1]



Spark: 3.5.0
Tables loaded: collisions, vehicles, casualties, collisions_enriched


## 3. Inspect rows, columns and data types

`.show(10)` displays ten rows, `.printSchema()` describes columns and their types, and `.count()` counts the table's rows. Row order is not guaranteed.

In [3]:
vehicles.select(
    "collision_index", "vehicle_reference", "vehicle_type_label",
    "age_of_driver", "age_of_driver_clean"
).show(10, truncate=False)

vehicles.printSchema()

for name, df in tables.items():
    print(f"{name} : {df.count():,} rows")

+---------------+-----------------+------------------+-------------+-------------------+
|collision_index|vehicle_reference|vehicle_type_label|age_of_driver|age_of_driver_clean|
+---------------+-----------------+------------------+-------------+-------------------+
|2021622100736  |1                |Car               |35           |35                 |
|2021010319369  |1                |Car               |25           |25                 |
|2021122100703  |2                |Car               |50           |50                 |
|2021471117441  |1                |Car               |28           |28                 |
|2021451064862  |1                |Car               |28           |28                 |
|2021421057659  |1                |Car               |62           |62                 |
|2021010308814  |3                |Car               |46           |46                 |
|2021131099362  |2                |Car               |-1           |NULL               |
|2021010305746  |2   

collisions : 513,801 rows
vehicles : 937,265 rows
casualties : 652,821 rows


collisions_enriched : 513,801 rows


## 4. Demonstrate age cleaning

`vehicles` contains the driver's age; `casualties` contains the casualty's age. Negative ages become `NULL` in the `_clean` columns. The original columns are retained.

The filters below only display examples; they do not delete any rows from the saved tables.

In [4]:
print("Drivers: examples of negative ages replaced with NULL")
vehicles.filter(F.col("age_of_driver") < 0).select(
    "collision_index", "vehicle_reference",
    "age_of_driver", "age_of_driver_clean"
).show(10, truncate=False)

print("Casualties: examples of negative ages replaced with NULL")
casualties.filter(F.col("age_of_casualty") < 0).select(
    "collision_index", "casualty_reference",
    "age_of_casualty", "age_of_casualty_clean"
).show(10, truncate=False)

Drivers: examples of negative ages replaced with NULL


+---------------+-----------------+-------------+-------------------+
|collision_index|vehicle_reference|age_of_driver|age_of_driver_clean|
+---------------+-----------------+-------------+-------------------+
|2021131099362  |2                |-1           |NULL               |
|2021010313435  |1                |-1           |NULL               |
|2021361058796  |5                |-1           |NULL               |
|2021201095102  |1                |-1           |NULL               |
|2021201133433  |1                |-1           |NULL               |
|2021991094033  |4                |-1           |NULL               |
|2021622100910  |2                |-1           |NULL               |
|2021010341731  |2                |-1           |NULL               |
|2021031048701  |4                |-1           |NULL               |
|2021010296144  |1                |-1           |NULL               |
+---------------+-----------------+-------------+-------------------+
only showing top 10 

## 5. Inspect cleaned speed limits and dates

`speed_limit_mph_clean` retains only the recognised values: **20, 30, 40, 50, 60 and 70 mph**. Other values become `NULL`. This is not a conversion to km/h.

`date_source` retains the original text; `date` is a Spark date. `hour`, `month` and `is_weekend` support time-based analysis.

In [5]:
print("Original and cleaned speed limits")
collisions.groupBy("speed_limit", "speed_limit_mph_clean").count().orderBy(
    "speed_limit"
).show(30, truncate=False)

print("Examples of temporal fields")
collisions.select(
    "collision_index", "date_source", "date", "time",
    "hour", "month", "is_weekend"
).show(10, truncate=False)

Original and cleaned speed limits


+-----------+---------------------+------+
|speed_limit|speed_limit_mph_clean|count |
+-----------+---------------------+------+
|-1         |NULL                 |3     |
|20         |20                   |88096 |
|30         |30                   |267325|
|40         |40                   |44750 |
|50         |50                   |22666 |
|60         |60                   |62703 |
|70         |70                   |28258 |
+-----------+---------------------+------+

Examples of temporal fields
+---------------+-----------+----------+-----+----+-----+----------+
|collision_index|date_source|date      |time |hour|month|is_weekend|
+---------------+-----------+----------+-----+----+-----+----------+
|202417H105824  |03/10/2024 |2024-10-03|16:06|16  |10   |false     |
|202517M115025  |29/11/2025 |2025-11-29|16:34|16  |11   |true      |
|202517L306225  |01/06/2025 |2025-06-01|22:07|22  |6    |true      |
|202417S104724  |20/04/2024 |2024-04-20|14:40|14  |4    |true      |
|202517S106525 

## 6. Demonstrate category decoding

The STATS19 guide provides readable labels. For example, `collision_severity_label` explains the code in `collision_severity`. The original code is retained.

In [6]:
collisions.select(
    "collision_severity", "collision_severity_label"
).distinct().orderBy("collision_severity").show(truncate=False)

collisions.select(
    "collision_index", "weather_conditions", "weather_conditions_label"
).show(10, truncate=False)

+------------------+------------------------+
|collision_severity|collision_severity_label|
+------------------+------------------------+
|1                 |Fatal                   |
|2                 |Serious                 |
|3                 |Slight                  |
+------------------+------------------------+

+---------------+------------------+------------------------+
|collision_index|weather_conditions|weather_conditions_label|
+---------------+------------------+------------------------+
|202417H105824  |1                 |Fine no high winds      |
|202517M115025  |1                 |Fine no high winds      |
|202517L306225  |1                 |Fine no high winds      |
|202417S104724  |1                 |Fine no high winds      |
|202517S106525  |1                 |Fine no high winds      |
|202517S208125  |1                 |Fine no high winds      |
|202517S210225  |1                 |Fine no high winds      |
|2024111449241  |1                 |Fine no high winds   

## 7. Explore enriched collisions

Vehicles and casualties are first grouped by collision, then joined to `collisions`. This preserves **one row per collision** while adding summary measures.

The mean driver age uses only known driver ages after cleaning.

In [7]:
enriched.select(
    "collision_index",
    "number_of_vehicles", "observed_vehicle_count",
    "number_of_casualties", "observed_casualty_count",
    "known_driver_age_count", "mean_recorded_driver_age",
    "fatal_casualty_count", "serious_casualty_count", "slight_casualty_count"
).show(10, truncate=False)

print("Collisions before enrichment:", report["tables"]["collisions"]["rows"])
print("Collisions after enrichment:", report["enriched_rows"])

+---------------+------------------+----------------------+--------------------+-----------------------+----------------------+------------------------+--------------------+----------------------+---------------------+
|collision_index|number_of_vehicles|observed_vehicle_count|number_of_casualties|observed_casualty_count|known_driver_age_count|mean_recorded_driver_age|fatal_casualty_count|serious_casualty_count|slight_casualty_count|
+---------------+------------------+----------------------+--------------------+-----------------------+----------------------+------------------------+--------------------+----------------------+---------------------+
|2021010287155  |1                 |1                     |1                   |1                      |0                     |NULL                    |1                   |0                     |0                    |
|2021010287163  |2                 |2                     |1                   |1                      |1                   

## 8. Inspect the quality report

The report is a **Python dictionary** loaded from `quality_report.json`. It includes missing keys, excess duplicate keys and relationship anomalies between tables.

A key occurring three times represents two excess rows. These checks do not automatically delete rows. The run inspected here has already been validated.

In [8]:
print("Report:", run_path / "quality_report.json")

for name, metrics in report["tables"].items():
    print(f"\n{name}")
    print("  Rows:", metrics["rows"])
    print("  Rows with an incomplete key:", metrics["missing_keys"])
    print("  Excess duplicate keys:", metrics["duplicate_key_excess"])

print("\nRelationships between tables:")
print(json.dumps(report["relationships"], indent=2, ensure_ascii=False))

print("\nValidation after writing and reloading Parquet:")
print(json.dumps(report["parquet_validation"], indent=2, ensure_ascii=False))

Report: /home/ubuntu/group16/project/data/silver/dev/runs/20261115T161713385306Z/quality_report.json

collisions
  Rows: 513801
  Rows with an incomplete key: 0
  Excess duplicate keys: 0

vehicles
  Rows: 937265
  Rows with an incomplete key: 0
  Excess duplicate keys: 0

casualties
  Rows: 652821
  Rows with an incomplete key: 0
  Excess duplicate keys: 0

Relationships between tables:
{
  "vehicles_without_collision": 0,
  "vehicles_year_mismatches": 0,
  "casualties_without_collision": 0,
  "casualties_year_mismatches": 0,
  "casualties_without_vehicle": 0,
  "observed_vehicle_count_mismatches": 0,
  "observed_casualty_count_mismatches": 0
}

Validation after writing and reloading Parquet:
{
  "collisions": {
    "rows": 513801,
    "schema_matches": true
  },
  "vehicles": {
    "rows": 937265,
    "schema_matches": true
  },
  "casualties": {
    "rows": 652821,
    "schema_matches": true
  },
  "collisions_enriched": {
    "rows": 513801,
    "schema_matches": true
  }
}


## 9. Close the session

Run this after the demonstration to release Spark resources. To explore the tables again afterwards, rerun the cells starting from step 2.

In [9]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
